# Identifying volcanoes on Venus

We're going to use images collected by the [Magellan mission](https://www2.jpl.nasa.gov/magellan/)
to build a classifier that can identify volcanoes on the surface of Venus. Magellan mapped
Venus by radar between 1990 and 1994 — the surface is permanently hidden beneath cloud, so
radar is the only way to see it.

## Where this data comes from

The raw images are on NASA's website, but we use the processed 110×110 pixel "chips"
prepared by Manuel Mena and hosted on Kaggle:

**https://www.kaggle.com/datasets/fmena14/volcanoesvenus**

**Downloading it yourself requires a free Kaggle account** — they gate dataset downloads
behind sign-in. You do *not* need one for this notebook: a prepared copy ships with the
repository. The link is here so you know exactly where the data came from and can go back to
the original if you want to.

## What ships with the repo, and why it differs

Kaggle distributes the chips as CSV — one row per image, 12,100 columns of pixel values as
text. That is about four bytes per pixel instead of one, so the training file alone is
283 MB, and `pandas` expands it to **646 MB in memory** on load. Neither fits comfortably
anywhere.

`data/venus/` instead holds the same images as compressed `uint8` arrays:

| | Kaggle CSV | here |
|---|---|---|
| training images | 283 MB | 34 MB |
| load time | ~6 s | ~0.2 s |
| memory once loaded | 646 MB | 81 MB |

The **test set is complete** (2,734 images). The **training set is a stratified subsample**
of 4,001 of the original 7,000, chosen to keep the class balance identical — about 14%
of the chips contain a volcano — and to preserve the rarer volcano `Type` categories.
`tools/build_venus_dataset.py` documents exactly how, if you want to rebuild it from the
Kaggle original.


Let's load the images and labels.


In [ ]:
import pandas as pd
import numpy as np
import jax.numpy as jnp

import matplotlib.pyplot as plt

In [ ]:
from time import time
import datetime

from flax import linen as nn

In [ ]:
import os

In [ ]:
import numpy as np
import pandas as pd

DATA = '../../data/venus/'

# images: (n, 12100) uint8 -- each row is a 110x110 chip, flattened
train_images = np.load(DATA + 'train_images.npz')['images']
test_images  = np.load(DATA + 'test_images.npz')['images']
train_labels = pd.read_csv(DATA + 'train_labels.csv')
test_labels  = pd.read_csv(DATA + 'test_labels.csv')

print('train', train_images.shape, ' test', test_images.shape)
print(f"positive fraction -- train {train_labels['Volcano?'].mean():.1%}, "
      f"test {test_labels['Volcano?'].mean():.1%}")
train_labels.head()


In [ ]:
image = train_images[82].reshape((110, 110))
plt.imshow(image, cmap='gray');


In [ ]:
train_labels.head()

It looks like `NaN`s populate the fields other than `Volvano?` when there aren't any volcanoes in the image.  Let's replace them with `0`s to save ourselves potential headaches in the future.

In [ ]:
train_labels.fillna(value=0, inplace=True)
test_labels.fillna(value=0, inplace=True)
train_labels.head()

👍

Now let's build our design matrices, rescale them, and collect our results.

In [ ]:
X_train = np.array(train_images, dtype=float).reshape(train_images.shape[0], 110, 110, 1)
X_test = np.array(test_images, dtype=float).reshape(test_images.shape[0], 110, 110, 1)

In [ ]:
X_train /= 255
X_test /= 255

We could treat this as a multi-class classification problem and learn how to count volcanoes, but for now let's just learn how to identify at least one volcano.

In [ ]:
target = 'Volcano?'
ncats = 2

Y_train = train_labels[target]
Y_test = test_labels[target]

In [ ]:
plt.imshow(X_train[63, :, :, 0], cmap='gray');

It looks like some images in the data set are corrupted.  Let's remove any images that have more than $10\%$ pixels at `0`.

In [ ]:
corrupted_train_images = np.mean(X_train == 0, axis=(1, 2)).flatten() > .1
corrupted_test_images = np.mean(X_test == 0, axis=(1, 2)).flatten() > .1
plt.imshow(X_train[corrupted_train_images][1, :, :, 0], cmap='gray');

In [ ]:
X_train = jnp.array(X_train[~corrupted_train_images])
X_test = jnp.array(X_test[~corrupted_test_images])
Y_train = jnp.array(Y_train[~corrupted_train_images].astype(int))
Y_test = jnp.array(Y_test[~corrupted_test_images].astype(int))

In [ ]:
batch_size = 64
n_batches = X_train.shape[0] // batch_size

X_train = X_train[:n_batches * batch_size].reshape((n_batches, batch_size, *X_train.shape[1:]))
Y_train = Y_train[:n_batches * batch_size].reshape((n_batches, batch_size, *Y_train.shape[1:]))

n_test_batches = X_test.shape[0] // batch_size
X_test = X_test[:n_test_batches * batch_size].reshape((n_test_batches, batch_size, *X_test.shape[1:]))
Y_test = Y_test[:n_test_batches * batch_size].reshape((n_test_batches, batch_size, *Y_test.shape[1:]))

X_train.shape, Y_train.shape

Now let's build a CNN and train it.

Below is one example of a successful CNN architecture from Behcet Senturk [on kaggle](https://www.kaggle.com/behcetsenturk/finding-volcanoes-with-cnn).

In [ ]:
# model = tf.keras.Sequential()
# model.add(layers.Conv2D(filters=2, kernel_size=(3, 3), padding='same',
#                  activation='relu', input_shape=(110, 110, 1)))
# model.add(layers.Conv2D(filters=4, kernel_size=(3, 3),
#                  padding='same', activation='relu'))
# model.add(layers.Conv2D(filters=8, kernel_size=(5, 5),
#                  padding='same',activation ='relu'))
# model.add(layers.MaxPool2D(pool_size=(2, 2)))
# model.add(layers.Dropout(0.5))
# model.add(layers.Conv2D(filters=16, kernel_size=(5,5),
#                  padding='same', activation ='relu'))
# model.add(layers.MaxPool2D(pool_size=(2, 2)))
# model.add(layers.Conv2D(filters=24, kernel_size=(7, 7),
#                  padding='same', activation='relu'))
# model.add(layers.Dropout(0.5))
# model.add(layers.Flatten())
# model.add(layers.Dense(Y_train.shape[1], activation="softmax"))

In [ ]:
class ConvNN(nn.Module):
  """A simple model with densely connected layers."""

  @nn.compact
  def __call__(self, x):
    x = nn.Conv(features=64, kernel_size=(3, 3))(x)
    x = nn.relu(x)
    x = x.reshape((x.shape[0], -1))
    x = nn.Dense(128)(x)
    x = nn.relu(x)
    x = nn.Dense(ncats)(x)
    return x

In [ ]:
import jax
import jax.numpy as jnp  # JAX NumPy
dummy_input = jnp.ones((1, 110, 110, 1))
cnn = ConvNN()
print(cnn.tabulate(jax.random.PRNGKey(0), dummy_input))

In [ ]:
init_rng = jax.random.PRNGKey(0)

In [ ]:
from clu import metrics
from flax.training import train_state
from flax import struct
import optax

@struct.dataclass
class Metrics(metrics.Collection):
    accuracy: metrics.Accuracy
    loss: metrics.Average.from_output('loss')

class TrainState(train_state.TrainState):
   metrics: Metrics

def create_train_state(model, rng, learning_rate):
    params = model.init(rng, dummy_input)['params']
    tx = optax.adam(learning_rate)
    return TrainState.create(
        apply_fn=model.apply, params=params, tx=tx,
        metrics=Metrics.empty())

@jax.jit
def train_step(state, batch, label):
  """Train for a single step."""
  def loss_fn(params):
    logits = state.apply_fn({'params': params}, batch)
    loss = optax.softmax_cross_entropy_with_integer_labels(
        logits=logits, labels=label).mean()
    return loss
  grad_fn = jax.grad(loss_fn)
  grads = grad_fn(state.params)
  state = state.apply_gradients(grads=grads)
  return state

@jax.jit
def compute_metrics(*, state, batch, label):
    logits = state.apply_fn({'params': state.params}, batch)
    loss = optax.softmax_cross_entropy_with_integer_labels(
        logits=logits, labels=label).mean()
    metric_updates = state.metrics.single_from_model_output(
        logits=logits, labels=label, loss=loss)
    metrics = state.metrics.merge(metric_updates)
    state = state.replace(metrics=metrics)
    return state

In [ ]:
init_rng = jax.random.PRNGKey(0)

learning_rate = 0.01

state = create_train_state(cnn, init_rng, learning_rate)
del init_rng  # Must not be used anymore.

In [ ]:
metrics_history = {'train_loss': [],
                   'train_accuracy': [],
                   'test_loss': [],
                   'test_accuracy': []}

In [ ]:
n_epochs = 3

In [ ]:
from tqdm import tqdm

In [ ]:
step = 0

for _ in range(n_epochs):
  for batch, label in tqdm(zip(X_train, Y_train), total=n_batches):

    # Run optimization steps over training batches and compute batch metrics
    state = train_step(state, batch, label) # get updated train state (which contains the updated parameters)
    state = compute_metrics(state=state, batch=batch, label=label) # aggregate batch metrics

    if (step+1) % n_batches == 0: # one training epoch has passed
      for metric,value in state.metrics.compute().items(): # compute metrics
        metrics_history[f'train_{metric}'].append(value) # record metrics
      state = state.replace(metrics=state.metrics.empty()) # reset train_metrics for next training epoch

      # Compute metrics on the test set after each training epoch
      test_state = state
      i = np.random.randint(X_test.shape[0])
      test_state = compute_metrics(state=test_state, batch=X_test[i], label=Y_test[i])

      for metric,value in test_state.metrics.compute().items():
        metrics_history[f'test_{metric}'].append(value)

      print(f"train epoch: {(step+1) // n_batches}, "
            f"loss: {metrics_history['train_loss'][-1]}, "
            f"accuracy: {metrics_history['train_accuracy'][-1] * 100}")
      print(f"test epoch: {(step+1) // n_batches}, "
            f"loss: {metrics_history['test_loss'][-1]}, "
            f"accuracy: {metrics_history['test_accuracy'][-1] * 100}")
    step += 1

In [ ]:
# Plot loss and accuracy in subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
ax1.set_title('Loss')
ax2.set_title('Accuracy')
for dataset in ('train','test'):
  ax1.plot(metrics_history[f'{dataset}_loss'], label=f'{dataset}_loss')
  ax2.plot(metrics_history[f'{dataset}_accuracy'], label=f'{dataset}_accuracy')
ax1.legend()
ax2.legend()
plt.show()
plt.clf()

In [ ]:
@jax.jit
def predict(state, batch):
  logits = state.apply_fn({'params': state.params}, batch)
  return logits.argmax(axis=1)

Now let's perform some of the same checks we've done in the past, e.g., generate a confusion matrix, look at some of the best and worst predictions, etc.

In [ ]:
state.apply_fn({'params': state.params}, X_test[0]), Y_test[0]

In [ ]:
def check(state, batch, labels):
    logits = state.apply_fn({'params': state.params}, batch)
    pred = predict(state, batch)

    confusion_matrix = np.zeros((ncats, ncats))
    for truth, guess in zip(labels, pred):
        confusion_matrix[truth, guess] += 1

    plt.imshow(confusion_matrix)
    plt.xlabel("Prediction")
    plt.ylabel("Truth")
    plt.show()

    print("Number of incorrect guesses: ", np.sum(pred != labels), " of ", len(batch))

    predictions_for_true_values = [logit[i] for logit, i in zip(logits, labels)]

    bins = plt.hist(predictions_for_true_values, bins=30, range=(0, 1))
    plt.title("Output for correct classes")

    volcano = np.arange(len(label))[label == 1]

    worst = np.argsort(predictions_for_true_values)
    plt.show()

    print("Worst:")
    for index in worst[:5]:
        print("Guess: ", pred[index], ' truth: ', labels[index])
        plt.imshow(np.squeeze(batch[index]), cmap='gray')
        plt.show()

    print("Best:")
    for index in worst[-5:]:
        print("Guess: ", pred[index], ' truth: ', labels[index])
        plt.imshow(np.squeeze(batch[index]), cmap='gray')
        plt.show()

In [ ]:
check(state, X_test[0], Y_test[0])